In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. โหลดข้อมูล
df_games = pd.read_csv('/content/all_games.csv')

df_games.isnull().sum()

,0
name,0
platform,0
release_date,0
summary,114
meta_score,0
user_review,0


In [24]:
(df_games['user_review'] == 'tbd').sum()

np.int64(1365)

In [25]:
# Data Cleaning
df_games = df_games.dropna(subset=['meta_score'])
df_games['summary'] = df_games['summary'].fillna('No summary')

df_games['user_review'] = pd.to_numeric(df_games['user_review'], errors='coerce')
review_median = df_games['user_review'].median()
# metascore x มี user_review เฉลี่ย y
mean_review_by_meta = df_games.groupby('meta_score')['user_review'].transform('mean')
# 3. เติมค่าว่าง (ที่เคยเป็น tbd) ด้วยค่าเฉลี่ยของกลุ่ม metascore นั้นๆ
df_games['user_review'] = df_games['user_review'].fillna(mean_review_by_meta)
# 4. กรณีที่ยังมีค่าว่างเหลืออยู่ (เช่น metascore นั้นไม่มีคู่เทียบเลย) ให้เติมด้วยค่าเฉลี่ยรวม
df_games['user_review'] = df_games['user_review'].fillna(review_median)

(df_games['user_review'] == 'tbd').sum()

np.int64(0)

In [26]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Feature Engineering & Encoding
# แปลง Platform เป็นตัวเลข (ใช้ df_games ให้ตรงกับข้างบน)
le_platform = LabelEncoder()
df_games['platform_encoded'] = le_platform.fit_transform(df_games['platform'])

# ดึงข้อมูล ปี และ เดือน ออกจาก release_date เพื่อเป็น Feature เพิ่มเติม
df_games['release_date'] = pd.to_datetime(df_games['release_date'])
df_games['year'] = df_games['release_date'].dt.year
df_games['month'] = df_games['release_date'].dt.month

# 4. เลือก Features ที่จะใช้ทำนาย meta_score
# เราจะใช้ Platform, User Review, Year, และ Month
X = pd.get_dummies(df_games[['platform', 'user_review', 'year', 'month']], columns=['platform'])
y = df_games['meta_score']

# 5. แบ่งข้อมูล Training และ Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scaling ข้อมูล (สำคัญมากสำหรับ SVR และ Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [27]:
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

# 1. นิยามโมเดลย่อย 3 ประเภท
model1 = RandomForestRegressor(n_estimators=100, random_state=42) # แบบ Tree
model2 = LinearRegression()                                      # แบบ Linear
model3 = SVR(kernel='rbf')                                       # แบบ Kernel

# 2. สร้าง Ensemble Model (มัดรวมกัน)
ensemble_model = VotingRegressor(estimators=[
    ('rf', model1),
    ('lr', model2),
    ('svr', model3)
])

# 3. ฝึกสอนโมเดล (ใช้ข้อมูลที่ Scale แล้ว)
ensemble_model.fit(X_train_scaled, y_train)

# 4. ทดสอบพยากรณ์
y_pred = ensemble_model.predict(X_test_scaled)

# 5. แสดงผลการวัดประสิทธิภาพ
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R-squared Score: {r2_score(y_test, y_pred):.2f}")

Mean Absolute Error (MAE): 7.08
R-squared Score: 0.40


In [28]:
import joblib

joblib.dump(ensemble_model, 'ensemble_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

model_columns = X.columns.tolist()
joblib.dump(model_columns, 'model_columns.pkl')

from google.colab import files
files.download('ensemble_model.pkl')
files.download('scaler.pkl')
files.download('model_columns.pkl')

df_games.to_csv('cleaned_all_games.csv', index=False)
files.download('cleaned_all_games.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>